# 06 - Create the Direct Lake semantic model

Run this **inside Fabric** (after notebooks 01-04 have built the `gold` tables). It
creates a **Direct Lake** semantic model over the gold tables and adds the business
relationships and DAX measures from `fabric/semantic-model/model_spec.yaml`.

**Steps:** attach your Telco Lakehouse as the notebook's default Lakehouse, then
**Run all**. The model reads Delta directly from OneLake (no import/refresh), so it
always reflects the latest notebook run.

> Direct Lake authoring uses `semantic-link-labs`, which runs in Fabric (not on a
> local workstation). The spec below mirrors `model_spec.yaml`, the source of truth.
> Steps are best-effort: a `sempy_labs` version mismatch logs and continues rather
> than aborting, and you can always finish in the portal / Tabular Editor.

In [ ]:
%pip install --quiet semantic-link-labs

## Model spec (mirrors fabric/semantic-model/model_spec.yaml)

In [ ]:
DATASET = 'TelcoCustomerService'
LAKEHOUSE_DEFAULT = 'TelcoLakehouse'  # fallback only
SCHEMA = 'gold'
TABLES = ['dim_customer', 'dim_account', 'dim_geography', 'dim_product', 'dim_plan', 'dim_promotion', 'fact_subscription', 'fact_invoice', 'fact_invoice_line', 'fact_usage_data', 'fact_service_metric', 'fact_outage', 'fact_work_order', 'fact_offer', 'fact_feedback', 'ml_churn_score', 'ml_crosssell_reco', 'customer_360']
RELATIONSHIPS = [{'from': 'dim_account.customer_id', 'to': 'dim_customer.customer_id'}, {'from': 'dim_customer.geo_id', 'to': 'dim_geography.geo_id'}, {'from': 'fact_subscription.account_id', 'to': 'dim_account.account_id'}, {'from': 'fact_subscription.product_id', 'to': 'dim_product.product_id'}, {'from': 'fact_subscription.plan_id', 'to': 'dim_plan.plan_id'}, {'from': 'fact_invoice.account_id', 'to': 'dim_account.account_id'}, {'from': 'fact_invoice_line.invoice_id', 'to': 'fact_invoice.invoice_id'}, {'from': 'fact_usage_data.account_id', 'to': 'dim_account.account_id'}, {'from': 'fact_service_metric.account_id', 'to': 'dim_account.account_id'}, {'from': 'fact_outage.geo_id', 'to': 'dim_geography.geo_id'}, {'from': 'fact_work_order.account_id', 'to': 'dim_account.account_id'}, {'from': 'fact_offer.account_id', 'to': 'dim_account.account_id'}, {'from': 'fact_offer.promotion_id', 'to': 'dim_promotion.promotion_id'}, {'from': 'fact_feedback.account_id', 'to': 'dim_account.account_id'}, {'from': 'ml_churn_score.customer_id', 'to': 'dim_customer.customer_id'}, {'from': 'ml_crosssell_reco.account_id', 'to': 'dim_account.account_id'}]
MEASURES = [{'table': 'dim_customer', 'name': 'Customer Count', 'expression': 'DISTINCTCOUNT(dim_customer[customer_id])', 'format': '#,0'}, {'table': 'dim_account', 'name': 'Active Accounts', 'expression': 'CALCULATE(DISTINCTCOUNT(dim_account[account_id]), dim_account[status] = "active")', 'format': '#,0'}, {'table': 'dim_account', 'name': 'Suspended Accounts', 'expression': 'CALCULATE(DISTINCTCOUNT(dim_account[account_id]), dim_account[status] = "suspended")', 'format': '#,0'}, {'table': 'fact_subscription', 'name': 'Monthly Recurring Revenue', 'expression': 'CALCULATE(SUM(fact_subscription[mrc]), fact_subscription[status] = "active")', 'format': '\\$#,0.00'}, {'table': 'fact_invoice', 'name': 'Total Billed', 'expression': 'SUM(fact_invoice[amount])', 'format': '\\$#,0.00'}, {'table': 'fact_invoice', 'name': 'Open Balance', 'expression': 'CALCULATE(SUM(fact_invoice[amount]), fact_invoice[paid] = FALSE())', 'format': '\\$#,0.00'}, {'table': 'fact_invoice', 'name': 'First Bill Count', 'expression': 'CALCULATE(COUNTROWS(fact_invoice), fact_invoice[is_first_bill] = TRUE())', 'format': '#,0'}, {'table': 'ml_churn_score', 'name': 'High Risk Customers', 'expression': 'CALCULATE(DISTINCTCOUNT(ml_churn_score[customer_id]), ml_churn_score[risk_band] = "High")', 'format': '#,0'}, {'table': 'ml_churn_score', 'name': 'Avg Churn Probability', 'expression': 'AVERAGE(ml_churn_score[churn_probability])', 'format': '0.0%'}, {'table': 'fact_service_metric', 'name': 'Avg Uptime Pct', 'expression': 'AVERAGE(fact_service_metric[uptime_pct])', 'format': '0.00'}, {'table': 'fact_feedback', 'name': 'Avg CSAT', 'expression': 'AVERAGE(fact_feedback[csat])', 'format': '0.0'}, {'table': 'fact_offer', 'name': 'Offer Acceptance Rate', 'expression': 'DIVIDE(\n  CALCULATE(COUNTROWS(fact_offer), fact_offer[status] = "accepted"),\n  COUNTROWS(fact_offer)\n)\n', 'format': '0.0%'}]

## Resolve the current workspace + attached Lakehouse

Uses whichever Lakehouse you attach as the notebook's default, so it works no matter
what you named it. `LAKEHOUSE_DEFAULT` is only a fallback.

In [ ]:
workspace_id = None
LAKEHOUSE = None
try:
    import notebookutils
    ctx = notebookutils.runtime.context
    workspace_id = ctx.get('currentWorkspaceId')
    LAKEHOUSE = ctx.get('defaultLakehouseName')
except Exception:
    pass
if not workspace_id:
    import sempy.fabric as fabric
    workspace_id = fabric.get_notebook_workspace_id()
if not LAKEHOUSE:
    LAKEHOUSE = LAKEHOUSE_DEFAULT
    print(f'WARNING: no default Lakehouse detected - falling back to {LAKEHOUSE!r}.')
print('Workspace:', workspace_id)
print('Lakehouse:', LAKEHOUSE)

## Create the Direct Lake model

Creates the model over the gold tables. If your Lakehouse is schema-enabled, the
tables live in the `gold` schema; pass schema-qualified names when the API supports it.

In [ ]:
import sempy_labs as labs
from sempy_labs import directlake

# API (current): generate_direct_lake_semantic_model(dataset, tables, source, ...)
# 'tables' are schema-qualified (gold.<name>); 'source' is the Lakehouse.
created = False
for tbls in ([f'{SCHEMA}.{t}' for t in TABLES], TABLES):
    try:
        directlake.generate_direct_lake_semantic_model(
            dataset=DATASET, tables=tbls, source=LAKEHOUSE,
            source_type='Lakehouse', workspace=workspace_id, overwrite=True)
        created = True
        print('Created Direct Lake model', DATASET, 'with', len(tbls), 'tables.')
        break
    except Exception as ex:
        print('attempt failed:', ex)
if not created:
    print('Could not auto-create the model. Create it in the portal: Lakehouse ->')
    print('New semantic model -> pick the gold tables, then re-run the cell below',
          'to add relationships + measures.')

## Add relationships + business measures

Applies the relationships and DAX measures from the spec via the Tabular Object Model.

In [ ]:
def _try(fn, label):
    try:
        fn(); print('  ok:', label)
    except Exception as ex:
        print('  skipped:', label, '(', ex, ')')

try:
    with labs.tom.connect_semantic_model(
            dataset=DATASET, workspace=workspace_id, readonly=False) as tom:
        for rel in RELATIONSHIPS:
            ft, fc = rel['from'].split('.'); tt, tc = rel['to'].split('.')
            _try(lambda ft=ft, fc=fc, tt=tt, tc=tc: tom.add_relationship(
                from_table=ft, from_column=fc, to_table=tt, to_column=tc,
                from_cardinality='Many', to_cardinality='One'),
                f"rel {rel['from']} -> {rel['to']}")
        for m in MEASURES:
            _try(lambda m=m: tom.add_measure(
                table_name=m['table'], measure_name=m['name'],
                expression=m['expression'].strip(), format_string=m.get('format')),
                f"measure {m['name']}")
    print('Relationships + measures applied.')
except Exception as ex:
    print('Could not connect to the model via TOM:', ex)
    print('Apply model_spec.yaml with Tabular Editor or the portal instead.')

## Validate

Lists the measures now on the model. It's ready for Power BI reports and for the
Fabric Data Agent / FabricIQ to consume as a business-friendly layer.

In [ ]:
try:
    import sempy.fabric as fabric
    dfM = fabric.list_measures(dataset=DATASET, workspace=workspace_id)
    print('measures on', DATASET, ':', len(dfM))
    display(dfM[['Table Name', 'Measure Name']] if len(dfM) else dfM)
except Exception as ex:
    print('list_measures unavailable:', ex)